In [ ]:
# analysis 4
import neuroimage_analysis as na
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Directories

In [ ]:
dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
outdir = os.path.join(dir, 'results')
schaefer_fc_matrix = np.load(os.path.join(dir, 'data/matrix/mean_adjacency_matrix_1000x1000.npy'))

## Analysis 4: Does sLNM Converge to the Degree Map or Gradient 1?

In real data, the degree map and gradient 1 of the reference connectome (C) are highly similar, making it difficult to determine which drives sLNM outputs. To dissociate them, we perform simulations where we explicitly manipulate which principal component (gradient) carries the most variance.

**Connectome manipulation via eigenswap**
We downsample from voxel space to 1,000 Schaefer parcels and eigendecompose C, yielding eigenvectors (gradients) and eigenvalues (proportion of variance). By swapping the first and k-th eigenvalues, we reassign maximum variance to any chosen component k, producing a randomized connectome C′.

**Simulation setup**
Each subject's FC map is set to the corresponding row of C′ (lesions constrained to a single Schaefer parcel). Symptoms are modeled from ground truth networks at η² = 0.99. All 1,000 Schaefer parcels are used as ground truth seeds in turn, yielding 1,000 sLNM maps per eigenswap.

**Identifying the dominant sLNM pattern**
We apply hierarchical clustering (average linkage, distance = 1 − |r|, threshold |r| > 0.3) to the 1,000 sLNM maps. The dominant pattern is defined as the sign-aligned average of maps in the largest cluster. We then compare this pattern to gradient 1 and the degree map of C′ using absolute Pearson correlation, across eigenswaps k = 1 to 75.

In [ ]:
def eigendecom(matrix):
    """Return eigenvalues and eigenvectors of a matrix sorted in descending order."""
    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    idx = np.argsort(eigenvalues)[::-1]
    return eigenvalues[idx], eigenvectors[:, idx]


def plot_matrix(matrix, title=None):
    """Plot a matrix using a custom diverging colormap."""
    import matplotlib.colors as mcolors

    cmap = mcolors.LinearSegmentedColormap.from_list(
        'custom',
        colors=['cyan', 'blue', 'black', 'red', 'orange', 'yellow'],
        N=256
    )
    plt.imshow(matrix, cmap=cmap, vmin=-0.4, vmax=0.5)
    plt.xticks([])
    plt.yticks([])
    if title is not None:
        plt.title(title)
    plt.show()

In [ ]:
plot_matrix(schaefer_fc_matrix, title = "GSP1000, Schaefer 1000-Parcellations")

# sLNM analyses

In [ ]:
def bootstrap_dataset(matrix, sample_size=100, eta_sq=0.99, gt_seed=None):
    """
    Generate a bootstrapped FC map and synthetic symptom score dataset for sLNM analysis.

    Randomly selects a ground truth connectome column, correlates bootstrapped
    subjects against it, then scales by effect size and adds noise to produce
    z-scored behavioral scores.

    Parameters
    ----------
    matrix      : (parcels x parcels) connectivity matrix
    sample_size : number of subjects to bootstrap (default: 100)
    eta_sq      : effect size as eta-squared (default: 0.99)
    gt_seed     : column index of ground truth map (random if None)

    Returns
    -------
    dict with 'ground_truth' (parcels,) and 'lnm' (parcels,)
    """
    # Select ground truth map
    if gt_seed is None:
        gt_seed = np.random.choice(matrix.shape[0])
    gt_map = matrix[:, gt_seed]

    # Bootstrap subjects and compute their FC maps
    subject_idx = np.random.choice(matrix.shape[0], sample_size, replace=True)
    subject_map = matrix[:, subject_idx]

    # Generate synthetic behavioral scores based on similarity to ground truth
    r_gt = na.pearson_rows(subject_map.T, gt_map)
    raw_scores = r_gt * np.sqrt(eta_sq / (1 - eta_sq))
    noisy_scores = raw_scores + np.random.normal(0, 1.0, len(r_gt))
    z_scores = (noisy_scores - np.mean(noisy_scores)) / np.std(noisy_scores)

    lnm_map = na.voxel_outcome_correlation(subject_map.T, z_scores)

    return {
        'ground_truth': gt_map,
        'lnm': lnm_map
    }


def main_analysis(matrix, sample_size=100, effect_size=0.99, seed=42):
    """
    Run the main sLNM convergence analysis on a connectivity matrix.

    For every possible ground truth seed parcel, generates an sLNM map via
    bootstrapping. Clusters the resulting maps by spatial similarity to identify
    the dominant convergent pattern, then compares it against degree and PC1
    (Gradient 1) of the connectome.

    Parameters
    ----------
    matrix      : (parcels x parcels) connectivity matrix
    sample_size : parcels bootstrapped per dataset (default: 100)
    effect_size : eta-squared effect size (default: 0.99)
    seed        : random seed for reproducibility (default: 42)

    Returns
    -------
    dict with 'degree', 'gradient', 'lnm', 'cluster_size'
    """
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    np.random.seed(seed)

    # Decompose connectome into gradients and compute degree map
    _, gradients = eigendecom(matrix)
    grad1 = gradients[:, 0]
    degree = np.sum(matrix, axis=0)

    # Generate one sLNM map per parcel (used as ground truth seed)
    lnm_maps = np.array([
        bootstrap_dataset(matrix, sample_size=sample_size, eta_sq=effect_size, gt_seed=i)['lnm']
        for i in range(matrix.shape[0])
    ])

    # Cluster sLNM maps by absolute spatial similarity to find dominant pattern
    lnm_corr = np.corrcoef(lnm_maps)
    distance = (1 - np.abs(lnm_corr))
    distance = (distance + distance.T) / 2  # ensure symmetry
    np.fill_diagonal(distance, 0)

    linkage_matrix = linkage(squareform(distance), method='average')

    threshold = 0.3  # maps with |r| > 0.3 are grouped together
    clusters = fcluster(linkage_matrix, t=(1 - threshold), criterion='distance')

    # Identify the largest cluster as the dominant convergent pattern
    labels, counts = np.unique(clusters, return_counts=True)
    dominant_cluster = labels[np.argmax(counts)]
    cluster_size = counts.max()
    print(f'Dominant sLNM pattern: {cluster_size} / {matrix.shape[0]} maps')

    # Average maps within dominant cluster, aligning signs before averaging
    cluster_maps = lnm_maps[clusters == dominant_cluster].copy()
    reference = cluster_maps[0]
    signs = np.sign([np.corrcoef(m, reference)[0, 1] for m in cluster_maps])
    lnm_pattern = (cluster_maps * signs[:, None]).mean(axis=0)

    # Align signs of grad1 and lnm_pattern for interpretability
    if pearsonr(grad1, degree)[0] < 0:
        grad1 = -grad1
    if pearsonr(grad1, lnm_pattern)[0] < 0:
        lnm_pattern = -lnm_pattern

    print(f'Degree  — Gradient 1 |r|: {np.abs(pearsonr(degree, grad1)[0]):.2f}')
    print(f'sLNM    — Degree     |r|: {np.abs(pearsonr(lnm_pattern, degree)[0]):.2f}')
    print(f'sLNM    — Gradient 1 |r|: {np.abs(pearsonr(lnm_pattern, grad1)[0]):.2f}')

    return {
        'degree': degree,
        'gradient': grad1,
        'lnm': lnm_pattern,
        'cluster_size': cluster_size
    }

In [ ]:
result = main_analysis(schaefer_fc_matrix, effect_size= 0.99)

# Randomize Gradients 

In [ ]:
def eigenswap(matrix, n_component):
    """
    Swap the eigenvalue of a target gradient with PC1 (Gradient 1).

    Used to test what sLNM would look like if a different gradient
    dominated the connectome's spectral structure.

    Parameters
    ----------
    matrix      : (parcels x parcels) connectivity matrix
    n_component : gradient to swap with PC1 (e.g. 2 swaps Gradient 2 with Gradient 1)

    Returns
    -------
    Reconstructed symmetric matrix with swapped eigenvalues
    """
    eigenvalues, eigenvectors = eigendecom(matrix)

    # Swap target eigenvalue with PC1 eigenvalue
    target = n_component - 1
    eigenvalues_swapped = eigenvalues.copy()
    eigenvalues_swapped[[0, target]] = eigenvalues_swapped[[target, 0]]

    # Reconstruct and symmetrize matrix
    new_matrix = eigenvectors @ np.diag(eigenvalues_swapped) @ eigenvectors.T
    new_matrix = (new_matrix + new_matrix.T) / 2

    return new_matrix

In [ ]:
# Example: swap Gradient k with Gradient 1 and run the main analysis
k = 10

swapped_matrix = eigenswap(schaefer_fc_matrix, n_component=k)
results_k = main_analysis(swapped_matrix)
# Visualize the matrix
plot_matrix(swapped_matrix, title=f'GSP1000, swapped {k}-th gradient')
# Visualize degree, gradient, and sLNM maps on the left hemisphere
for map_name, map_data in results_k.items():
    if isinstance(map_data, np.ndarray):  # skip scalar values like cluster_size
        na.recon_tmap(map_data, map_name, 'left', 'lateral')

# Numerical Results

In [ ]:
# Run eigenswap analysis across k = 1 to n components
n = 75

deg_grad_corr = []
lnm_grad_corr = []
lnm_deg_corr  = []
cluster_sizes  = []

for k in range(1, n + 1):
    print(f'Analysis k = {k}')
    swapped_matrix = eigenswap(schaefer_fc_matrix, n_component=k)
    results_k = main_analysis(swapped_matrix)

    degree = results_k['degree']
    grad   = results_k['gradient']
    lnm    = results_k['lnm']

    deg_grad_corr.append(np.abs(pearsonr(degree, grad)[0]))
    lnm_grad_corr.append(np.abs(pearsonr(grad, lnm)[0]))
    lnm_deg_corr.append(np.abs(pearsonr(degree, lnm)[0]))
    cluster_sizes.append(results_k['cluster_size'])

cluster_sizes = np.array(cluster_sizes)
print(f'Mean cluster size: {np.mean(cluster_sizes):.1f} ± {np.std(cluster_sizes):.1f}')

In [ ]:
# Plot spatial correlations across swapped components
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 10

fig, ax = plt.subplots(figsize=(10, 4), dpi=150)

x = range(n)
ax.plot(x, deg_grad_corr, label='Degree–Gradient 1', color='#888888', linewidth=2, linestyle='--')
ax.plot(x, lnm_grad_corr, label='sLNM–Gradient 1',  color='#c44e52', linewidth=2)
ax.plot(x, lnm_deg_corr,  label='sLNM–Degree',       color='#4c72b0', linewidth=2)

ax.set_xlabel('Swapped Component (k)')
ax.set_ylabel('|Spatial Correlation|')
ax.set_ylim(0, 1.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(frameon=True, fontsize=8, loc='lower left', facecolor='white', framealpha=0.6)

plt.tight_layout()
plt.show()